In [1]:
# 1:Cài đặt và Khởi tạo môi trường Spark
# Cài đặt pyspark nếu chưa có sẵn
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("IoT_Building_Diagnostic_TV1_EDA") \
    .getOrCreate()

spark

In [4]:
# 2: Kiểm chứng số lượng dòng thực tế & Chứng minh giới hạn Excel
# Kết nối với drive
from google.colab import drive
drive.mount('/content/drive')

# 2: Kiểm chứng số lượng dòng thực tế & Chứng minh giới hạn Excel
# Đọc file dữ liệu CSV
file_path = "/content/drive/MyDrive/11_iot_building_diagnostic.csv"

df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)

# 1. Đếm số lượng dòng bằng Spark count
total_rows = df.count()
print(f"Tổng số dòng thực tế trong Spark DataFrame: {total_rows:,} dòng")

# 2. Đối chiếu với giới hạn của Excel
excel_limit = 1048576
if total_rows > excel_limit:
    print(f"-> CHỨNG MINH: Số lượng dòng ({total_rows:,}) lớn hơn mức giới hạn tối đa của Excel ({excel_limit:,} dòng).")
    print(f"-> Nếu mở trực tiếp bằng Microsoft Excel, file sẽ bị cắt cụt mất {total_rows - excel_limit:,} dòng dữ liệu cuối!")
else:
    print("-> Số lượng dòng nằm trong giới hạn của Excel.")

Mounted at /content/drive
Tổng số dòng thực tế trong Spark DataFrame: 2,100,000 dòng
-> CHỨNG MINH: Số lượng dòng (2,100,000) lớn hơn mức giới hạn tối đa của Excel (1,048,576 dòng).
-> Nếu mở trực tiếp bằng Microsoft Excel, file sẽ bị cắt cụt mất 1,051,424 dòng dữ liệu cuối!


In [5]:
# 3: Phân tích thống kê mô tả & Phân bố nhãn mục tiêu (diagnostic_state)
print("--- 1. KIỂM TRA SCHEMA VÀ KIỂU DỮ LIỆU ---")
df.printSchema()

print("--- 2. PHÂN BỐ NHÃN MỤC TIÊU (DIAGNOSTIC_STATE) ---")
# Đánh giá hiện tượng mất cân bằng lớp (class imbalance)
label_dist = df.groupBy("diagnostic_state").count().withColumn(
    "percentage", F.round(F.col("count") * 100.0 / total_rows, 2)
)
label_dist.show()

--- 1. KIỂM TRA SCHEMA VÀ KIỂU DỮ LIỆU ---
root
 |-- reading_id: integer (nullable = true)
 |-- recorded_at: timestamp (nullable = true)
 |-- building_zone: string (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- floor_number: integer (nullable = true)
 |-- occupancy_count: integer (nullable = true)
 |-- outdoor_temp_c: double (nullable = true)
 |-- outdoor_humidity_pct: double (nullable = true)
 |-- outdoor_pm25_ug_m3: double (nullable = true)
 |-- indoor_temp_c: double (nullable = true)
 |-- indoor_humidity_pct: double (nullable = true)
 |-- co2_ppm: double (nullable = true)
 |-- pm25_ug_m3: double (nullable = true)
 |-- tvoc_ppb: double (nullable = true)
 |-- air_flow_m3_h: double (nullable = true)
 |-- fan_speed_rpm: double (nullable = true)
 |-- hvac_power_kw: double (nullable = true)
 |-- filter_pressure_pa: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- maintenance_days: integer (nullable = true)
 |-- window_open_pct: double (nullable 

In [6]:
# 4: Thống kê tỷ lệ giá trị khuyết (Missing Values)
print("--- 1. TỶ LỆ GIÁ TRỊ KHUYẾT TRÊN TỪNG CỘT ---")
exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
null_counts_df = df.agg(*exprs)
null_counts_df.show(vertical=True)

print("--- 2. TỶ LỆ GIÁ TRỊ KHUYẾT THEO TỪNG KHU VỰC (BUILDING_ZONE) ---")
# Kiểm tra xem giá trị khuyết có tập trung ở khu vực nào cụ thể không
null_by_zone = df.groupBy("building_zone").agg(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ["occupancy_count", "tvoc_ppb", "vibration_mm_s", "window_open_pct"]]
)
null_by_zone.show()

--- 1. TỶ LỆ GIÁ TRỊ KHUYẾT TRÊN TỪNG CỘT ---
-RECORD 0---------------------
 reading_id           | 0     
 recorded_at          | 0     
 building_zone        | 0     
 sensor_id            | 0     
 floor_number         | 0     
 occupancy_count      | 18996 
 outdoor_temp_c       | 0     
 outdoor_humidity_pct | 0     
 outdoor_pm25_ug_m3   | 0     
 indoor_temp_c        | 0     
 indoor_humidity_pct  | 0     
 co2_ppm              | 0     
 pm25_ug_m3           | 0     
 tvoc_ppb             | 33731 
 air_flow_m3_h        | 0     
 fan_speed_rpm        | 0     
 hvac_power_kw        | 0     
 filter_pressure_pa   | 0     
 vibration_mm_s       | 37969 
 maintenance_days     | 0     
 window_open_pct      | 23323 
 equipment_age_years  | 0     
 firmware_version     | 0     
 diagnostic_state     | 0     

--- 2. TỶ LỆ GIÁ TRỊ KHUYẾT THEO TỪNG KHU VỰC (BUILDING_ZONE) ---
+-------------+---------------+--------+--------------+---------------+
|building_zone|occupancy_count|tvoc_ppb|

In [7]:
# 5: Xử lý cột nhiễu, loại bỏ cột không dùng & Chia tập Train/Test theo thời gian (80/20)
print("--- 1. XỬ LÝ CỘT KHÓA VÀ KIỂM TRA CỘT NHIỄU ---")
# Loại bỏ cột reading_id để tránh overfitting
# Kiểm tra phân bố nhãn theo firmware_version để đánh giá mức độ nhiễu
print("Phân bố nhãn theo firmware_version:")
df.groupBy("firmware_version", "diagnostic_state").count().orderBy("firmware_version").show(20)

cleaned_df = df.drop("reading_id")

print("--- 2. CHIẾN LƯỢC CHIA TẬP TRAIN / TEST THEO THỜI GIAN (80/20) ---")
# Sắp xếp theo recorded_at và thêm cột thứ tự hàng (row_number)
windowSpec = Window.orderBy("recorded_at")
df_with_row = cleaned_df.withColumn("row_num", F.row_number().over(windowSpec))

# Xác định mốc cắt 80%
split_threshold = int(total_rows * 0.8)

train_df = df_with_row.filter(F.col("row_num") <= split_threshold).drop("row_num")
test_df = df_with_row.filter(F.col("row_num") > split_threshold).drop("row_num")

print(f"Số lượng bản ghi tập Train (80% thời gian đầu): {train_df.count():,}")
print(f"Số lượng bản ghi tập Test (20% thời gian sau): {test_df.count():,}")

# Checkpoint lưu dữ liệu sạch đã phân chia (nếu cần lưu ra parquet/csv trên Drive để Thành viên khác dùng)
# train_df.write.mode("overwrite").parquet("/content/drive/MyDrive/iot_data_processed/train_data.parquet")
# test_df.write.mode("overwrite").parquet("/content/drive/MyDrive/iot_data_processed/test_data.parquet")
print("-> Hoàn thành thiết lập chiến lược chia Train/Test ngăn ngừa Data Leakage thành công!")

--- 1. XỬ LÝ CỘT KHÓA VÀ KIỂM TRA CỘT NHIỄU ---
Phân bố nhãn theo firmware_version:
+----------------+-----------------+------+
|firmware_version| diagnostic_state| count|
+----------------+-----------------+------+
|          v2.4.1|ventilation_issue| 93355|
|          v2.4.1|           normal|395173|
|          v2.4.1|    thermal_issue| 36472|
|          v2.4.2|           normal|394117|
|          v2.4.2|ventilation_issue| 94006|
|          v2.4.2|    thermal_issue| 36877|
|          v2.5.0|ventilation_issue| 94629|
|          v2.5.0|           normal|393383|
|          v2.5.0|    thermal_issue| 36988|
|          v2.5.1|ventilation_issue| 94032|
|          v2.5.1|    thermal_issue| 36417|
|          v2.5.1|           normal|394551|
+----------------+-----------------+------+

--- 2. CHIẾN LƯỢC CHIA TẬP TRAIN / TEST THEO THỜI GIAN (80/20) ---
Số lượng bản ghi tập Train (80% thời gian đầu): 1,680,000
Số lượng bản ghi tập Test (20% thời gian sau): 420,000
-> Hoàn thành thiết lập chiến l

In [8]:
# Lưu tập Train và Test xuống thư mục trên Google Drive của bạn
train_output_path = "/content/drive/MyDrive/iot_train_data.csv"
test_output_path = "/content/drive/MyDrive/iot_test_data.csv"

# Xuất ra file CSV (hoặc parquet) để các thành viên khác dễ dàng đọc lại
train_df.write.mode("overwrite").option("header", "true").csv(train_output_path)
test_df.write.mode("overwrite").option("header", "true").csv(test_output_path)

print("Đã lưu checkpoint tập Train và Test lên Google Drive thành công!")

Đã lưu checkpoint tập Train và Test lên Google Drive thành công!
